# 📊 06｜縣市用電：把寬表轉成長表，才看得見時間

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/06_power_reshape/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：月份不是一個欄位，而是**分散在 24 個欄名**裡（寬表）；要先「轉成長表」，讓「月份」變成真正的時間欄位，才能畫趨勢、依年份分組。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：看懂寬表、轉成長表並核對總量，最後用**一張折線圖**回答「臺北市 24 個月的用電如何變化？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：匯出整理好的長表，供之後的樞紐分析或其他工具使用。
3. **選做挑戰（單元五～六）**：比較各區域兩年的用電，或用「比例」比較規模差很大的縣市的季節型態。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/power_wide.csv`（請先閱讀[本例資料卡](./data/DATA_CARD.md)）。原始寬表**一列代表：一個縣市的 24 個月用電**；轉成長表後，**一列代表：一個縣市、一個月份的用電量**。數值為教學用合成資料，單位為百萬度。

### 1. 欄位角色貼標籤
- 🔵 **條件 (Dimension)**：`縣市`、`區域`（類別）；以及**藏在 24 個欄名裡的年月**（時間）➔ 要轉成長表才變成真正的欄位。
- 🟢 **指標 (Metric)**：每個月份格子裡的數值（百萬度）➔ 轉長表後集中成一個「用電量」欄位。
- ⚪ **標識 (ID)**：本例沒有流水號；`縣市` 名稱在這裡當作分組用的條件。

### 2. 五大提問積木與核心問題
- 📈 **趨勢（必做）**：臺北市 2024～2025 年 24 個月的用電如何變化？哪幾個月最高？
- 📊 **比較（選做）**：各區域 2025 年的總用電比 2024 年多還是少？
- 🔍 **趨勢＋比例（選做）**：規模差很多的縣市（如新北市與連江縣），季節變化的型態一樣嗎？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **寬表無法直接回答時間問題**：時間藏在欄名裡，必須先轉成長表；轉換後列數應為 22 × 24 ＝ 528，總用電量要和寬表一致。
- ⚠️ **年月是文字**：「2024-01」要轉成日期，折線圖才會依時間正確排序。
- ⚠️ **總量不等於效率**：資料沒有人口或戶數，不能說「某縣市比較浪費電」；只能比較總用電量。
- ⚠️ **規模差很大時看比例**：連江縣約只有新北市的 0.2%，放在同一個數值軸上會變成一條平線，看不出它的季節變化。
- ⚠️ **兩年不足以預測**：24 個月只能描述季節與兩年差異，不能據此預測未來用電。

### 4. 決定最適圖表
- **單一縣市 24 個月走勢** ➔ **折線圖**（X 軸月份由早到晚；Y 軸標示百萬度）。
- **5 個區域兩年總量比較（選做）** ➔ **分組長條圖**（每區兩根柱子：2024、2025；Y 軸從 0 開始）。
- **規模不同縣市的季節型態（選做）** ➔ 兩張並排的**折線圖**：左邊原始數值、右邊換算成「占該縣市月平均的比例」。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'power_wide.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/06_power_reshape/data/power_wide.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取寬表，找出藏在欄名裡的時間（貼標籤）

> 🎯 **思維重點**：先看資料「長什麼樣子」：一列是一個縣市，24 個月份排在右邊。時間不是一個欄位，而是欄名。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/power_wide.csv：
1. 顯示前 5 筆資料、列數與欄數。
2. 列出所有欄位名稱，並說明哪些欄名其實是「時間」。
3. 檢查各欄缺值數。
4. 用文字說明：目前一列代表什麼？如果想畫「某縣市隨時間的變化」，為什麼這個格式不方便？
先不要轉換資料，也不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

wide = pd.read_csv('data/power_wide.csv')
print(f"寬表大小：{wide.shape[0]} 列 × {wide.shape[1]} 欄（一列 = 一個縣市）")
display(wide.head())

month_cols = [c for c in wide.columns if c not in ['縣市', '區域']]
print(f"\n條件欄位：['縣市', '區域']")
print(f"時間藏在欄名裡：共 {len(month_cols)} 個，從 {month_cols[0]} 到 {month_cols[-1]}")
print("缺值總數：", wide.isna().sum().sum())

### 📝 單元二【必做】：邊界檢核：寬表轉長表，核對列數與總量

> 🎯 **思維重點**：用 melt 把 24 個月份欄位「轉直」，變成「年月」與「用電量」兩欄；轉換是改變形狀，不是改變數字，所以總量必須一模一樣。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請把寬表轉成長表：
1. 保留「縣市」「區域」兩欄，把 24 個月份欄位轉成兩欄：「年月」（原本的欄名）與「用電量_百萬度」（原本的數值）。
2. 把「年月」轉成日期格式，另外拆出「年」與「月」兩個欄位。
3. 核對：長表列數是否等於 縣市數 × 月份數？長表的用電量總和是否等於寬表所有月份數值的總和？
4. 顯示長表前 5 列，並說明現在一列代表什麼。
不要修改原始 CSV 檔。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
long_df = wide.melt(id_vars=['縣市', '區域'], var_name='年月', value_name='用電量_百萬度')
long_df['日期'] = pd.to_datetime(long_df['年月'], format='%Y-%m')
long_df['年'] = long_df['日期'].dt.year
long_df['月'] = long_df['日期'].dt.month

print(f"長表列數：{len(long_df)}；縣市數 × 月份數 = {wide['縣市'].nunique()} × {len(month_cols)} = {wide['縣市'].nunique() * len(month_cols)}")
wide_total = wide[month_cols].sum().sum()
long_total = long_df['用電量_百萬度'].sum()
print(f"寬表總用電量：{wide_total:,.1f}；長表總用電量：{long_total:,.1f}；一致：{abs(wide_total - long_total) < 1e-6}")
print("➡️ 現在一列代表：一個縣市、一個月份的用電量")
display(long_df.head())

### 📝 單元三【必做】：提問積木【趨勢】— 臺北市 24 個月的用電如何變化？（折線圖）

> 🎯 **思維重點**：「時間 ＋ 指標」問演變走向，用【折線圖】；X 軸是轉換後的日期欄位，由早到晚排序，Y 軸標示「百萬度」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請從長表篩選「臺北市」，畫出 24 個月的用電折線圖：
1. X 軸為日期（由早到晚），Y 軸為用電量（百萬度），每個月加上圓點標記。
2. 標出用電最高與最低的月份與數值。
3. 圖表標題、座標軸使用繁體中文，並註明資料期間 2024-01～2025-12。
4. 用文字描述季節型態，並提醒：24 個月只能描述過去，不能預測未來。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

taipei = long_df[long_df['縣市'] == '臺北市'].sort_values('日期')
peak = taipei.loc[taipei['用電量_百萬度'].idxmax()]
low = taipei.loc[taipei['用電量_百萬度'].idxmin()]

plt.figure(figsize=(10, 5))
plt.plot(taipei['日期'], taipei['用電量_百萬度'], marker='o', color='#3498db', linewidth=2)
for point, color in [(peak, '#e74c3c'), (low, '#2c3e50')]:
    plt.scatter(point['日期'], point['用電量_百萬度'], color=color, s=90, zorder=3)
    plt.annotate(f"{point['年月']}\n{point['用電量_百萬度']:.1f}", (point['日期'], point['用電量_百萬度']),
                 textcoords='offset points', xytext=(0, 10), ha='center', color=color)
plt.ylim(0, taipei['用電量_百萬度'].max() * 1.15)
plt.title('臺北市每月用電量（2024-01～2025-12）', fontsize=14, fontweight='bold')
plt.xlabel('月份')
plt.ylabel('用電量（百萬度）')
plt.grid(linestyle='--', alpha=0.5)
plt.show()
print("💡 每年 7～8 月最高、2 月最低，兩年的季節形狀相似。")
print("⚠️ 邊界提醒：24 個月只能描述過去的季節型態，不能據此預測未來用電。")

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能指出寬表中「時間藏在欄名」，並說出轉長表後一列代表什麼。
- [ ] 能展示長表為 528 列，且長表總用電量＝寬表總用電量；折線圖 X 軸依日期由早到晚。
- [ ] 能用一句話解釋：為什麼不能從這份資料說「臺北市比較浪費電」？
- 💬 **討論題**：資料中沒有哪個欄位，所以不能比較「誰比較省電」？

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 寬表 22 列 × 26 欄；長表 **528 列**（22 縣市 × 24 個月），兩者總用電量相同。
- 臺北市折線圖顯示每年夏季（7～8 月）最高、冬季（2 月）最低；兩年的形狀相似。

</details>

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：匯出整理好的長表

> 🎯 **思維重點**：長表是「分析友善」的格式：之後要做樞紐分析、篩選或交給其他工具都更方便。匯出後同樣要核對列數與總量。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請把長表匯出成 Excel：
1. 欄位依序為：縣市、區域、年、月、年月、用電量_百萬度；依縣市、日期排序。
2. 存為 output/縣市用電_長表.xlsx（若 output 目錄不存在請自動建立）。
3. 匯出後重新讀取，核對列數為 528、用電量總和與寬表一致。
4. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
export_df = long_df.sort_values(['縣市', '日期'])[['縣市', '區域', '年', '月', '年月', '用電量_百萬度']]
os.makedirs('output', exist_ok=True)
output_path = 'output/縣市用電_長表.xlsx'
export_df.to_excel(output_path, index=False)

check = pd.read_excel(output_path)
print(f"重新讀取：{len(check)} 列；用電量總和 {check['用電量_百萬度'].sum():,.1f}（寬表 {wide_total:,.1f}）")

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 檔案！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【比較】— 各區域兩年的總用電（分組長條圖）

> 🎯 **思維重點**：有了「年」欄位，才能依年份分組。各區域縣市數不同，比較的是「總量」，不代表效率。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用長表比較各區域 2024 與 2025 年的總用電量：
1. 依區域與年加總用電量，整理成「列＝區域、欄＝年」的表格，並計算 2025 相對 2024 的增減百分比。
2. 畫分組長條圖：X 軸為區域，每區兩根柱子（2024、2025），Y 軸從 0 開始，單位百萬度，加上圖例。
3. 註明各區域包含的縣市數，並提醒：資料沒有人口數，不能比較用電效率。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
region_year = long_df.pivot_table(index='區域', columns='年', values='用電量_百萬度', aggfunc='sum')
region_year['增減(%)'] = ((region_year[2025] / region_year[2024] - 1) * 100).round(1)
region_year['縣市數'] = wide.groupby('區域')['縣市'].count()
region_year = region_year.sort_values(2025, ascending=False)
display(region_year.round(1))

ax = region_year[[2024, 2025]].plot.bar(figsize=(9, 5), color=['#a9cce3', '#1f618d'], rot=0, width=0.7)
ax.set_ylim(0, region_year[2025].max() * 1.15)
ax.set_title('各區域年度總用電量（2024 vs 2025）', fontsize=13, fontweight='bold')
ax.set_xlabel('區域')
ax.set_ylabel('年度總用電量（百萬度）')
ax.legend(title='年')
plt.show()
print("⚠️ 邊界提醒：各區域縣市數不同、資料沒有人口數，只能比較總量，不能說哪區比較浪費電。")

### 📝 單元六【選做挑戰】：提問積木【趨勢＋比例】— 規模差很大的縣市，季節型態一樣嗎？（兩張折線圖）

> 🎯 **思維重點**：連江縣只有新北市的千分之二；放在同一軸上會變平線。改成「每月用電 ÷ 該縣市 24 個月平均」的比例，才能比較形狀。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請比較新北市、臺中市、連江縣三個縣市的季節型態：
1. 左圖：三條折線畫原始用電量（百萬度），觀察連江縣是否幾乎看不見變化。
2. 右圖：把每個月的用電量除以該縣市 24 個月的平均，換算成「相對月平均的比例（%）」，再畫三條折線，加上 100% 參考線。
3. 兩張圖並排、各自加上標題、軸名與圖例，全部使用繁體中文。
4. 用文字說明：換算成比例後，三個縣市的季節型態是否相似？比例圖能不能用來比較誰用電多？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
cities = ['新北市', '臺中市', '連江縣']
subset = long_df[long_df['縣市'].isin(cities)].sort_values('日期').copy()
subset['相對月平均(%)'] = subset['用電量_百萬度'] / subset.groupby('縣市')['用電量_百萬度'].transform('mean') * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for city in cities:
    part = subset[subset['縣市'] == city]
    axes[0].plot(part['日期'], part['用電量_百萬度'], marker='.', label=city)
    axes[1].plot(part['日期'], part['相對月平均(%)'], marker='.', label=city)
axes[0].set_title('原始用電量：連江縣幾乎貼在底部', fontsize=12, fontweight='bold')
axes[0].set_ylabel('用電量（百萬度）')
axes[1].axhline(100, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('換算成「相對自己月平均」的比例', fontsize=12, fontweight='bold')
axes[1].set_ylabel('相對月平均（%）')
for ax in axes:
    ax.set_xlabel('月份')
    ax.legend()
    ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()
print("💡 換算成比例後，三個縣市的夏高冬低型態相似；但比例圖只比較形狀，不能用來比較誰用電多。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：寬表轉長表 528 列且總量一致；臺北市折線圖依日期排序，看出夏高冬低。
- [ ] **報表應用**：長表 Excel 重新讀取後列數與總量正確。
- [ ] **選做挑戰**：區域比較只談總量；比例折線圖只比較季節形狀。
- [ ] **AI 協作**：能說明自己提出的問題、AI 幫忙執行的事，以及自己如何確認結果。